# holdout-data-one-per-class — ex2: seeded random one-per-class holdout for a stable gallery

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `holdout-data-one-per-class`. Running the final beacon cell reports progress against the `Generative: Hold-out one-per-class data` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Hold-out one-per-class data` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`holdout-data-one-per-class`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "holdout-data-one-per-class"
DD_SUBTOPIC = "Generative: Hold-out one-per-class data"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Deterministic random per-class holdout — quick refresher

ex1 picked the FIRST sample of each class — cheap, but biased toward whatever the data loader happens to surface first. A more representative gallery uses a RANDOM sample per class, sampled with a fixed seed so the gallery is stable across runs:
```python
g = t.Generator().manual_seed(seed)
for c in range(num_classes):
    idxs_for_c = (labels == c).nonzero(as_tuple=True)[0]
    pick = idxs_for_c[t.randint(len(idxs_for_c), (1,), generator=g)]
    holdout.append(data[pick.item()])
```

**Why a generator (not global seed).** The global RNG state changes every time any other code samples — your gallery would silently shift. A local `t.Generator()` keeps the selection reproducible regardless of surrounding training noise.

**`nonzero(as_tuple=True)[0]`** is the canonical recipe for 'integer indices where mask is True'. Equivalent to `t.where(mask)[0]`. The result is a 1-D `int64` tensor of positions you can sample from.

### Exercise 2 — seeded random one-per-class holdout for a stable gallery

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a SEEDED local `t.Generator` to pick a random index per class via `(labels == c).nonzero(as_tuple=True)[0]`, then stack into a reproducible holdout gallery.
> Keywords: holdout, random-pick, seeded-generator, nonzero
> ```

**KCs targeted:** `nonzero-mask-to-indices`, `seeded-generator-for-reproducibility`

Implement `ex2_seeded_random_per_class(data, labels, num_classes, seed)`. Like ex1, but instead of taking the FIRST sample of each class, pick a deterministic-random one using a local seeded RNG:

1. Build a local generator: `g = t.Generator().manual_seed(seed)`. Use a LOCAL generator (not the global RNG) so the pick is stable regardless of surrounding `torch.manual_seed` calls.
2. For each class `c` in `range(num_classes)`:
   - Find indices where `labels == c`: `idxs = (labels == c).nonzero(as_tuple=True)[0]`. This is a 1-D `int64` tensor of positions in `labels` whose value is `c`.
   - Sample one index from `idxs` using the generator: `pick = t.randint(len(idxs), (1,), generator=g).item()`. This gives an index INTO `idxs`, so the actual data index is `idxs[pick].item()`.
   - Append `data[idxs[pick].item()]` to the per-class list.
3. Stack: `return t.stack(per_class, dim=0)`. Shape `(num_classes, *sample_shape)`.

Assume every class has at least one sample (no need to handle empty `idxs`).

Input: `data` `(N, *)`, `labels` `(N,)` int64, `num_classes` int, `seed` int.
Output: `(num_classes, *)` tensor.

The visualization renders the seeded gallery and confirms that two calls with the same seed produce the IDENTICAL gallery, while different seeds typically produce different ones.

In [ ]:
def ex2_seeded_random_per_class(data: Tensor, labels: Tensor,
                                num_classes: int, seed: int) -> Tensor:
    g = t.Generator().manual_seed(seed)
    per_class = []
    for c in range(num_classes):
        idxs = (labels == c).nonzero(as_tuple=True)[0]
        pick = t.randint(len(idxs), (1,), generator=g).item()
        per_class.append(data[idxs[pick].item()])
    return t.stack(per_class, dim=0)


<details><summary>Solution</summary>

```python
def ex2_seeded_random_per_class(data: Tensor, labels: Tensor,
                                num_classes: int, seed: int) -> Tensor:
    g = t.Generator().manual_seed(seed)
    per_class = []
    for c in range(num_classes):
        idxs = (labels == c).nonzero(as_tuple=True)[0]
        pick = t.randint(len(idxs), (1,), generator=g).item()
        per_class.append(data[idxs[pick].item()])
    return t.stack(per_class, dim=0)
```

**Local generator is the only way to be reproducible.** The global RNG is shared with every other piece of torch code in your script — data augmentation, dropout, weight init. Any of those moving by a tick reorders the global state, which silently shifts your gallery. A local `t.Generator()` is insulated.

**`nonzero(as_tuple=True)[0]` vs `nonzero()`.** Tuple form returns one 1-D tensor per axis of the input. For a 1-D mask the tuple has length 1, so `[0]` extracts it. The non-tuple form returns a `(K, 1)` 2-D tensor — usable, but awkward.

**Why `t.randint(len(idxs), (1,))` not `t.randperm(len(idxs))[0]`.** Both correct, but `randint` is `O(1)` while `randperm` is `O(N)`. For a tight per-class loop on many classes, the difference is measurable.

**Imbalanced robustness.** When a class has only one sample, `t.randint(1, ...)` returns `0` deterministically — the unique sample. The test's imbalanced case asserts this; no special case in the code.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()